In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=200
    )

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type="torch", columns=["input_ids","label"])
test_data.set_format(type="torch", columns=["input_ids","label"])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

class TextCNN(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=128, num_classes=2):
        super(TextCNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=100,
            kernel_size=5
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(kernel_size=2)

        self.fc = nn.Linear(100*98, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        x = x.permute(0,2,1)

        x = self.conv(x)

        x = self.relu(x)

        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TextCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):

    model.train()

    total_loss = 0

    for batch in train_loader:

        inputs = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(train_loader))

    model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        inputs = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        outputs = model(inputs)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100*correct/total)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch: 1 Loss: 0.6093913365507979
Epoch: 2 Loss: 0.3052151736843845
Epoch: 3 Loss: 0.11608895111609907
Accuracy: 78.836
